In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import wandb
import random

from datasets import load_dataset
import evaluate
from dataclasses import dataclass, asdict
from transformers import (
    AutoTokenizer,
    AutoModel,
    BartTokenizer,
    BartForConditionalGeneration,
)
from transformers.modeling_outputs import BaseModelOutput

# =====================
# Config
# =====================
@dataclass
class TrainingConfig:
    batch_size: int = 8   # updated
    lr: float = 2e-5
    num_epochs: int = 10  # updated
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log_interval: int = 10
    max_len: int = 512

config = TrainingConfig()

# =====================
# Init WandB
# =====================
wandb.init(
    project="Prot",
    config=asdict(config),
    name="bart_multimodal_stable"
)

# =====================
# Dataset
# =====================
dataset = load_dataset("vladak/drug_protein_mechanism")

# =====================
# Tokenizers
# =====================
chem_tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
prot_tokenizer = AutoTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

# =====================
# Models
# =====================
chem_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1").to(config.device)
prot_model = AutoModel.from_pretrained("Rostlab/prot_bert").to(config.device)
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(config.device)

fusion_dim = bart_model.config.d_model
fusion_layer = nn.Linear(
    chem_model.config.hidden_size + prot_model.config.hidden_size,
    fusion_dim
).to(config.device)

# =====================
# Dataset processing
# =====================
def collate_fn(batch):
    smiles = [x["drug_smiles"] for x in batch]
    prots = [x["target_sequence"] for x in batch]
    texts = [x["mechanistic_explanation"] for x in batch]

    chem_enc = chem_tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
    prot_enc = prot_tokenizer(prots, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
    text_enc = bart_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)

    return chem_enc, prot_enc, text_enc, texts

train_loader = DataLoader(dataset["train"], batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset["val"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset["test"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)

# =====================
# Optimizer
# =====================
optimizer = torch.optim.AdamW(
    list(fusion_layer.parameters()) + list(bart_model.parameters()), 
    lr=config.lr
)

# =====================
# Metrics
# =====================
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_text_metrics(preds, refs):
    results = {}
    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    rouge = rouge_metric.compute(predictions=preds, references=refs)
    bert = bertscore_metric.compute(predictions=preds, references=refs, lang="en")

    results["bleu"] = bleu["score"]
    results["rougeL"] = rouge["rougeL"]
    results["bertscore"] = sum(bert["f1"]) / len(bert["f1"])
    return results

def count_parameters(*models):
    total_params = sum(p.numel() for m in models for p in m.parameters())
    trainable_params = sum(p.numel() for m in models for p in m.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params, trainable_params = count_parameters(chem_model, prot_model, fusion_layer, bart_model)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


# =====================
# Pick 4 fixed validation sample indices
# =====================
num_val_samples = len(dataset["val"])
fixed_val_indices = random.sample(range(num_val_samples), 4)

# =====================
# Training Loop
# =====================
for epoch in range(config.num_epochs):
    bart_model.train()
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

    for step, (chem_enc, prot_enc, text_enc, texts) in enumerate(progress):
        optimizer.zero_grad()

        # Encode SMILES
        chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
        chem_emb = chem_out.last_hidden_state.mean(dim=1)

        # Encode Protein
        prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
        prot_emb = prot_out.last_hidden_state.mean(dim=1)

        # Fuse
        fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))

        # Wrap in BaseModelOutput for BART
        encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

        labels = text_enc["input_ids"].to(config.device)

        outputs = bart_model(
            encoder_outputs=encoder_outputs,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if step % config.log_interval == 0:
            avg_loss = total_loss / (step + 1)
            progress.set_postfix({"loss": avg_loss})
            wandb.log({"train_loss": avg_loss, "epoch": epoch+1, "step": step})

    # =====================
    # Validation
    # =====================
    bart_model.eval()
    preds, refs = [], []

    with torch.no_grad():
        all_val_samples = []
        for chem_enc, prot_enc, text_enc, texts in tqdm(val_loader, desc="Validation"):
            chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
            chem_emb = chem_out.last_hidden_state.mean(dim=1)

            prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
            prot_emb = prot_out.last_hidden_state.mean(dim=1)

            fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
            encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
            encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

            # ✅ Deterministic decoding
            generated = bart_model.generate(
                encoder_outputs=encoder_outputs,
                max_length=config.max_len,
                num_beams=4,       # beam search for stability (use 1 for greedy)
                do_sample=False,   # turn off randomness
                early_stopping=True
            )
            decoded = bart_tokenizer.batch_decode(generated, skip_special_tokens=True)

            preds.extend(decoded)
            refs.extend(texts)
            all_val_samples.extend(list(zip(texts, decoded)))

        # Compute metrics
        metrics = compute_text_metrics(preds, refs)
        print(f"Validation metrics: {metrics}")
        wandb.log({f"val_{k}": v for k, v in metrics.items()})

        # Print fixed 4 validation samples predictions
        print("\nValidation sample predictions:")
        for idx in fixed_val_indices:
            sample = dataset["val"][idx]
            # encode & predict this sample only
            chem_enc = chem_tokenizer([sample["drug_smiles"]], return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
            prot_enc = prot_tokenizer([sample["target_sequence"]], return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
            chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
            chem_emb = chem_out.last_hidden_state.mean(dim=1)
            prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
            prot_emb = prot_out.last_hidden_state.mean(dim=1)
            fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
            encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
            encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

            generated = bart_model.generate(encoder_outputs=encoder_outputs, max_length=config.max_len)
            pred_text = bart_tokenizer.decode(generated[0], skip_special_tokens=True)

            print("REF: ", sample["mechanistic_explanation"])
            print("PRED:", pred_text)
            print("-" * 80)

# =====================
# Test
# =====================
bart_model.eval()
preds, refs = [], []
with torch.no_grad():
    for chem_enc, prot_enc, text_enc, texts in tqdm(test_loader, desc="Testing"):
        chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
        chem_emb = chem_out.last_hidden_state.mean(dim=1)

        prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
        prot_emb = prot_out.last_hidden_state.mean(dim=1)

        fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
        encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

        # ✅ Deterministic decoding
        generated = bart_model.generate(
            encoder_outputs=encoder_outputs,
            max_length=config.max_len,
            num_beams=4,       # beam search for stability (use 1 for greedy)
            do_sample=False,   # turn off randomness
            early_stopping=True
        )
        decoded = bart_tokenizer.batch_decode(generated, skip_special_tokens=True)

        preds.extend(decoded)
        refs.extend(texts)

metrics = compute_text_metrics(preds, refs)
print(f"Test metrics: {metrics}")
wandb.log({f"test_{k}": v for k, v in metrics.items()})



wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: upravljac (bgi9999). Use `wandb login --relogin` to force relogin


Total parameters: 604,832,512
Trainable parameters: 604,832,512


Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:19<00:00,  1.97s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Validation metrics: {'bleu': 11.338783095582095, 'rougeL': 0.25696178538063985, 'bertscore': 0.8544460009587439}

Validation sample predictions:
REF:  The drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below. In the biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug interacts with M2.
PRED: The drug DRUG interacts with its target TARGET (TARGET) The drug interacts with the GABAergic receptor in the text below. It provides an indication of its potential role in the development of the drug TARGET.
--------------------------------------------------------------------------------
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaco

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:13<00:00,  1.35s/it]


...

Validation metrics: {'bleu': 78.4587976784037, 'rougeL': 0.8309775687837107, 'bertscore': 0.9652414212101385}

Validation sample predictions:
REF:  The drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below. In the biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug interacts with M2.
PRED: The drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below. In the biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug interacts with M2.
--------------------------------------------------------------------------------
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAac

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:15<00:00,  1.56s/it]


Validation metrics: {'bleu': 79.5826565828524, 'rougeL': 0.8229576466436799, 'bertscore': 0.9645598875848871}

Validation sample predictions:
REF:  The drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below. In the biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug interacts with M2.
PRED:  DRUG interacts with its target TARGET (TARGET) AChEIs cause whole-body up-regulation of cholinergic signalling, resulting in dose-limiting adverse effects, including nausea, diarrhoea, salivation, cramping, and reduced heart rate. NMDA receptor antagonists, including memantine slows the decline in cognitive function in patients with moderate to severe AD.
--------------------------------------------------------------------------------
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGE

Testing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:14<00:00,  1.49s/it]


Test metrics: {'bleu': 85.45020913720941, 'rougeL': 0.8809299935545194, 'bertscore': 0.9757850951962657}
